# Authors & contributors from Dataverse JSON in MongoDB

Extracts every author and contributor from `datasetVersion.metadataBlocks.citation.fields`
into a pandas DataFrame (one row per person).

Requires the data to be loaded first with `json-to-mongo.py load <dir>`.

In [1]:
# !pip install pymongo pandas
import pandas as pd
from pymongo import MongoClient

MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "json_inspect"
COLL_NAME = "docs"

coll = MongoClient(MONGO_URI)[DB_NAME][COLL_NAME]
print("Documents in collection:", coll.count_documents({}))

Documents in collection: 9578


## Aggregation pipeline

In [2]:
BASE = "datasetVersion.metadataBlocks.citation.fields"

pipeline = [
    # Pre-filter: only docs that have an author or contributor field
    {"$match": {f"{BASE}.typeName": {"$in": ["author", "contributor"]}}},
    # One doc per element of the fields array, keep authors/contributors
    {"$unwind": f"${BASE}"},
    {"$match": {f"{BASE}.typeName": {"$in": ["author", "contributor"]}}},
    # One doc per person inside the field's value array
    {"$unwind": f"${BASE}.value"},
    # Name the long paths once
    {"$set": {"role": f"${BASE}.typeName", "p": f"${BASE}.value"}},
    {"$project": {
        "_id": 0,
        "_source_file": 1,
        "isAuthor": {"$eq": ["$role", "author"]},
        "name": {"$ifNull": ["$p.authorName.value", "$p.contributorName.value"]},
        "contributorType":        "$p.contributorType.value",
        "authorAffiliation":      "$p.authorAffiliation.value",
        "authorIdentifierScheme": "$p.authorIdentifierScheme.value",
        "authorIdentifier":       "$p.authorIdentifier.value",
    }},
]

## Run it and build the DataFrame

In [3]:
df = pd.DataFrame(coll.aggregate(pipeline, allowDiskUse=True))

# Make sure all expected columns exist even if no row has them
cols = ["_source_file", "isAuthor", "name", "contributorType",
        "authorAffiliation", "authorIdentifierScheme", "authorIdentifier"]
df = df.reindex(columns=cols)

print(df.shape)
df.head(10)

(23462, 7)


,_source_file,isAuthor,name,contributorType,authorAffiliation,authorIdentifierScheme,authorIdentifier
0,metadata_demo.dataverse.org/doi_10.5072_FK2_U6...,True,"Castro, Eleni",NaN,Harvard,ORCID,14252638xx
1,metadata_demo.dataverse.org/doi_10.5072_FK2_PP...,True,"Finlay, Jocelyn",NaN,Harvard University,ORCID,1234
2,metadata_demo.dataverse.org/doi_10.70122_FK2_H...,True,"Beasley, Will",NaN,Univ of Oklahoma,NaN,NaN
3,metadata_demo.dataverse.org/doi_10.5072_FK2_Q1...,True,"Andreev, Leonid",NaN,BARI/Dataverse,NaN,NaN
4,metadata_demo.dataverse.org/doi_10.5072_FK2_HJ...,True,"Andreev, Leonid",NaN,BARI/Dataverse,NaN,NaN
5,metadata_demo.dataverse.org/doi_10.70122_FK2_P...,True,"Kuriwaki, Shiro",NaN,Harvard University,ORCID,0000-0002-5687-2647
6,metadata_demo.dataverse.org/doi_10.5072_FK2_XS...,True,"Admin, Dataverse",NaN,Dataverse.org,ORCID,NaN
7,doi_10.34894_W3QRSV.dataverse_json.json,True,"O'Connor, Rachel",NaN,leidenuniv.nl,NaN,NaN
8,doi_10.34894_7IWSEX.dataverse_json.json,True,Esther Jansma,NaN,Stichting RING,ORCID,0000-0003-1690-2976
9,doi_10.34894_7IWSEX.dataverse_json.json,True,Elsemieke Spoor-Hanraets,NaN,Stichting RING,NaN,NaN


## Quick exploration

In [4]:
print("Authors:     ", int(df.isAuthor.sum()))
print("Contributors:", int((~df.isAuthor).sum()))
print("Files with at least one person:", df._source_file.nunique())

Authors:      22502
Contributors: 960
Files with at least one person: 9578


In [5]:
# Contributors by type
df[~df.isAuthor].contributorType.value_counts(dropna=False)

contributorType
Researcher             210
Data Collector         186
Supervisor             116
Project Member         107
Project Leader          97
Data Curator            53
Data Manager            34
Work Package Leader     26
NaN                     24
Other                   22
Related Person          18
Research Group          16
Funder                  15
Project Manager         15
Editor                   7
Hosting Institution      7
Sponsor                  4
Rights Holder            3
Name: count, dtype: int64

In [6]:
# Most common author affiliations
df[df.isAuthor].authorAffiliation.value_counts().head(20)

authorAffiliation
Stichting RING                                                        4962
Maastricht University                                                 1136
University of Groningen                                                791
Leiden University                                                      621
BAAC bv                                                                472
Tilburg University                                                     450
https://ror.org/057w15z03                                              425
Utrecht University                                                     384
Vrije Universiteit Amsterdam                                           375
Van Daalen Dendrochronologie                                           350
https://ror.org/012p63287                                              293
Rijksdienst voor het Oudheidkundig Bodemonderzoek (nu: RCE)            277
BAAC                                                                   272
Royal I

In [7]:
# Authors that have an identifier (e.g. ORCID)
df[df.isAuthor & df.authorIdentifier.notna()][
    ["_source_file", "name", "authorIdentifierScheme", "authorIdentifier"]
].head(20)

,_source_file,name,authorIdentifierScheme,authorIdentifier
0,metadata_demo.dataverse.org/doi_10.5072_FK2_U6...,"Castro, Eleni",ORCID,14252638xx
1,metadata_demo.dataverse.org/doi_10.5072_FK2_PP...,"Finlay, Jocelyn",ORCID,1234
5,metadata_demo.dataverse.org/doi_10.70122_FK2_P...,"Kuriwaki, Shiro",ORCID,0000-0002-5687-2647
8,doi_10.34894_7IWSEX.dataverse_json.json,Esther Jansma,ORCID,0000-0003-1690-2976
11,doi_10.34894_O7MZG8.dataverse_json.json,"van Goor, Harriet",ORCID,0000-0002-6332-1325
13,doi_10.34894_IHQLXY.dataverse_json.json,Tamara Vernimmen,ORCID,0000-0003-0813-177X
14,doi_10.34894_IHQLXY.dataverse_json.json,Esther Jansma,ORCID,0000-0003-1690-2976
16,doi_10.34894_GZMOKZ.dataverse_json.json,Tamara Vernimmen,ORCID,0000-0003-0813-177X
17,doi_10.34894_GZMOKZ.dataverse_json.json,Esther Jansma,ORCID,0000-0003-1690-2976
19,doi_10.34894_GKZHT5.dataverse_json.json,"Lloyd, Beth",ORCID,0000-0003-4034-8119


In [8]:
# Most frequent people across all files
df.groupby("name").size().sort_values(ascending=False).head(20)

name
Esther Jansma               2372
Elsemieke Spoor-Hanraets    1392
Tamara Vernimmen             697
Marta Domínguez Delmás       558
Daalen, S.                   487
van Daalen, Sjoerd           321
Doeve, Petra                 288
Fraiture, PF                 225
Tegel, WT                    122
Dieuwertje Duijn             120
Lambert, GL                   96
Pauline van Rijn              91
Haneca, Kristof               86
Benno Ridderhof               81
Dirk Jan de Vries             74
Hans Tisje                    73
Ute Sass-Klaassen             69
Jansma, E.                    50
J.A. Brongers                 47
Stapel, Rombert               44
dtype: int64

## Export (optional)

In [ ]:
from pathlib import Path

# Notebook lives in src/exploring/json-metadata/, so the repo root is 3 levels up
out_dir = Path("../../../data").resolve()
out_dir.mkdir(exist_ok=True)
df.to_csv(out_dir / "people.csv", index=False)
print("Wrote", out_dir / "people.csv")